In [ ]:
# Noah Lebowitz-Lockard
# Sunday, May 10, 2026

# final_birdclef_model

import librosa
import numpy as np
import pandas as pd
import pickle
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

In [ ]:
possible_test_dirs = [
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
    "/kaggle/input/datasets/noahlebowitzlockard/test-soundscapes"
]

TEST_DIR = None

for d in possible_test_dirs:
    if os.path.exists(d):
        TEST_DIR = d
        break

if TEST_DIR is None:
    raise FileNotFoundError("Could not find test_soundscapes directory.")

MODEL_DIR = "/kaggle/input/datasets/noahlebowitzlockard/birdclef-models"

In [ ]:
def build_animal_model(input_shape=(128, 313, 1), num_classes=5):
    model = tf.keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(num_classes, activation="sigmoid")
    ])
    return model

In [ ]:
def build_common_bird_model(input_shape=(128, 313, 1), num_classes=112):
    inputs = layers.Input(shape=input_shape)
    x = inputs

    for filters in [64, 128, 256, 256]:
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)

        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)

        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.2)(x)

    # Hybrid pooling
    x_avg = layers.GlobalAveragePooling2D()(x)
    x_max = layers.GlobalMaxPooling2D()(x)
    x = layers.Concatenate()([x_avg, x_max])

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)
    return model

In [ ]:
def build_common_amphibian_model(input_shape=(128, 313, 1), num_classes=14):
    inputs = tf.keras.Input(shape=input_shape)
    
    x = layers.Conv2D(32, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(256, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    return model

In [ ]:
def build_common_mammal_model(input_shape=(128, 313, 1), num_classes=5):
    model = tf.keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),

        layers.Dense(num_classes, activation="sigmoid")
    ])
    return model

In [ ]:
def build_common_insect0_model(input_shape=(128, 313, 1)):
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.Conv2D(16, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(32, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

In [ ]:
def build_common_insect1_model(input_shape=(128, 313, 1), num_classes=15):
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.Conv2D(32, (3,3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(64, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(128, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(256, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x_avg = layers.GlobalAveragePooling2D()(x)
    x_max = layers.GlobalMaxPooling2D()(x)
    x = layers.Concatenate()([x_avg, x_max])

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)
    return model

In [ ]:
animal_model = build_animal_model()
animal_model.load_weights(f"{MODEL_DIR}/best_multilabel_model.weights.h5")

common_bird_model = build_common_bird_model()
common_bird_model.load_weights(f"{MODEL_DIR}/common_bird_model.weights.h5")

common_amphibian_model = build_common_amphibian_model()
common_amphibian_model.load_weights(f"{MODEL_DIR}/common_amphibian_model.weights.h5")

common_mammal_model = build_common_mammal_model()
common_mammal_model.load_weights(f"{MODEL_DIR}/common_mammal_model.weights.h5")

common_insect0_model = build_common_insect0_model()
common_insect0_model.load_weights(f"{MODEL_DIR}/common_insect0_model.weights.h5")

common_insect1_model = build_common_insect1_model()
common_insect1_model.load_weights(f"{MODEL_DIR}/common_insect1_model.weights.h5")

In [ ]:
mammal_feature_model = tf.keras.Model(
    inputs=common_mammal_model.inputs,
    outputs=common_mammal_model.layers[-2].output
)

bird_feature_model = tf.keras.Model(
    inputs=common_bird_model.inputs,
    outputs=common_bird_model.layers[-2].output
)

insect_feature_model = tf.keras.Model(
    inputs=common_insect1_model.inputs,
    outputs=common_insect1_model.layers[-2].output
)

amphibian_feature_model = tf.keras.Model(
    inputs=common_amphibian_model.inputs,
    outputs=common_amphibian_model.layers[-2].output
)

In [ ]:
# Here are all of the animal species. The only reptile is 116570.

common_amphibians = ['1491113', '22961', '22967', '22973', '23158', '24279', '24321', '25092', '326272', '517063', '555146', '65377', '65380']
common_amphibians += ['66971']
rare_amphibians = ['1176823', '1595929', '22930', '22956', '22983', '22985', '23150', '23154', '23176', '23724', '24285', '24287', '25073']
rare_amphibians += ['25214', '476521', '555123', '555145', '64898', '67107', '67252', '70711']
common_birds = ['astcra1', 'baffal1', 'banana', 'barant1', 'batbel1', 'baymac', 'bbwduc', 'bkcdon', 'blheag1', 'bncfly', 'bobfly1', 'brcmar1']
common_birds += ['brnowl', 'bucmot4', 'bucpar', 'bufpar', 'bunibi1', 'burowl', 'camfli1', 'chacha1', 'chbmoc1', 'chobla1', 'chvcon1', 'coffal1']
common_birds += ['compau', 'compot1', 'crbthr1', 'epaori4', 'eulfly1', 'fepowl', 'flawar1', 'fusfly1', 'gilhum1', 'giwrai1', 'grasal3', 'greani1']
common_birds += ['greant1', 'greela', 'grekis', 'gretho2', 'greyel', 'grfdov1', 'gycwor1', 'houspa', 'larela1', 'lesela1', 'limpki', 'linwoo1']
common_birds += ['litnig1', 'masgna1', 'oliwoo1', 'orwpar', 'osprey', 'pabspi1', 'paltan1', 'phecuc1', 'picpig2', 'pirfly1', 'plasla1', 'plcjay1']
common_birds += ['pvttyr1', 'rebscy1', 'recfin1', 'redjun', 'relser1', 'rinkin1', 'roahaw', 'rubthr1', 'rufgna3', 'rufhor2', 'rufnig1', 'rumfly1']
common_birds += ['rutjac1', 'saffin', 'saytan1', 'scadov1', 'schpar1', 'shcfly1', 'shtnig1', 'sibtan2', 'smbani', 'sobcac1', 'sobtyr1']
common_birds += ['socfly1', 'sofspi1', 'souant1', 'soulap1', 'spbant3', 'spispi1', 'squcuc1', 'stbwoo2', 'strcuc1', 'strher2', 'strowl1']
common_birds += ['swtman1', 'tattin1', 'thlwre1', 'toctou1', 'trokin', 'trsowl', 'undtin1', 'varant1', 'watjac1', 'wesfie1', 'wfwduc1', 'whbwar2']
common_birds += ['whiwoo1', 'whtdov', 'y00678', 'yebela1', 'yecpar', 'yeofly1']
rare_birds = ['ashgre1', 'bafcur1', 'bcwfin2', 'bkhpar', 'blchaw1', 'blttit1', 'cibspi1', 'crebec1', 'dwatin1', 'fabwre1', 'ficman1', 'fotfly']
rare_birds += ['glteme1', 'grepot1', 'grhtan1', 'horscr1', 'hyamac1', 'lesgrf1', 'litcuc2', 'mabpar', 'magant1', 'magtan2', 'nacnig1', 'ocecra1']
rare_birds += ['orbtro3', 'palhor3', 'platyr1', 'pluibi1', 'purjay1', 'ragmac1', 'rivwar1', 'rufcac2', 'rufcas2', 'ruftho1', 'ruftof1']
rare_birds += ['ruther1', 'sabspa1', 'scther1', 'shshaw', 'smbtin1', 'souscr1', 'sptnig1', 'swthum1', 'whbant2', 'whlspi1', 'whnjay1', 'whwpic1']
rare_birds += ['yebcar', 'yecmac', 'yehcar1']

# common_insects excludes 244024 because it has a separate model.

common_insects = ['47158son01', '47158son03', '47158son04', '47158son06', '47158son07', '47158son08', '47158son10', '47158son11', '47158son13']
common_insects += ['47158son17', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25']
rare_insects = ['1161364', '47158son02', '47158son05', '47158son09', '47158son12', '47158son14', '47158son15', '47158son16', '47158son18']
rare_insects += ['47158son19', '47158son20', '760266']

common_mammals = ['41970', '43435', '47144', '516975', '74113']
rare_mammals = ['209233', '738183', '74580']

In [ ]:
import pickle

with open(f"{MODEL_DIR}/rare_embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

with open(f"{MODEL_DIR}/rare_base_rates.pkl", "rb") as f:
    rare_base_rates = pickle.load(f)

with open(f"{MODEL_DIR}/rare_centers.pkl", "rb") as f:
    rare_centers = pickle.load(f)

with open(f"{MODEL_DIR}/rare_sharpnesses.pkl", "rb") as f:
    rare_sharpnesses = pickle.load(f)

In [ ]:
# Using the similarity score and base rate, we estimate the actual probability that a given rare animal is present in the spectrogram.

def similarity_to_probability(score, base_rate, center, sharpness):
    prior_logit = np.log(base_rate / (1 - base_rate))

    evidence_logit = sharpness * (score - center)

    prob = 1 / (1 + np.exp(-(prior_logit + evidence_logit)))

    return prob

In [ ]:
# This block predicts how likely it is that a spectrogram belongs to each animal species.
# The classes are Insecta, Reptilia, Amphibia, Mammalia, Aves.

def predict_batch(X_file):
    batch_preds = []

    animal_preds = animal_model.predict(X_file, verbose=0)
    common_bird_preds = common_bird_model.predict(X_file, verbose=0)
    common_amphibian_preds = common_amphibian_model.predict(X_file, verbose=0)
    common_insect0_preds = common_insect0_model.predict(X_file, verbose=0)
    common_insect1_preds = common_insect1_model.predict(X_file, verbose=0)
    common_mammal_preds = common_mammal_model.predict(X_file, verbose=0)

    bird_embs = bird_feature_model.predict(X_file, verbose=0)
    amphibian_embs = amphibian_feature_model.predict(X_file, verbose=0)
    insect_embs = insect_feature_model.predict(X_file, verbose=0)
    mammal_embs = mammal_feature_model.predict(X_file, verbose=0)

    bird_embs = bird_embs / (np.linalg.norm(bird_embs, axis=1, keepdims=True) + 1e-8)
    amphibian_embs = amphibian_embs / (np.linalg.norm(amphibian_embs, axis=1, keepdims=True) + 1e-8)
    insect_embs = insect_embs / (np.linalg.norm(insect_embs, axis=1, keepdims=True) + 1e-8)
    mammal_embs = mammal_embs / (np.linalg.norm(mammal_embs, axis=1, keepdims=True) + 1e-8)

    for j in range(len(X_file)):
        final_preds = {}

        animal_p = animal_preds[j]

        for i in range(len(common_insects)):
            final_preds[common_insects[i]] = float(animal_p[0] * common_insect1_preds[j, i])

        for i in range(len(common_amphibians)):
            final_preds[common_amphibians[i]] = float(animal_p[2] * common_amphibian_preds[j, i])

        for i in range(len(common_mammals)):
            final_preds[common_mammals[i]] = float(animal_p[3] * common_mammal_preds[j, i])

        for i in range(len(common_birds)):
            final_preds[common_birds[i]] = float(animal_p[4] * common_bird_preds[j, i])

        final_preds["244024"] = float(animal_p[0] * common_insect0_preds[j, 0])
        final_preds["116570"] = float(animal_p[1])

        for insect in rare_insects:
            score = np.max(np.dot(embeddings[insect], insect_embs[j]))
            p = similarity_to_probability(score, rare_base_rates[insect], rare_centers[insect], rare_sharpnesses[insect])
            final_preds[insect] = float(animal_p[0] * p)

        for amphibian in rare_amphibians:
            score = np.max(np.dot(embeddings[amphibian], amphibian_embs[j]))
            p = similarity_to_probability(score, rare_base_rates[amphibian], rare_centers[amphibian], rare_sharpnesses[amphibian])
            final_preds[amphibian] = float(animal_p[2] * p)

        for mammal in rare_mammals:
            score = np.max(np.dot(embeddings[mammal], mammal_embs[j]))
            p = similarity_to_probability(score, rare_base_rates[mammal], rare_centers[mammal], rare_sharpnesses[mammal])
            final_preds[mammal] = float(animal_p[3] * p)

        for bird in rare_birds:
            score = np.max(np.dot(embeddings[bird], bird_embs[j]))
            p = similarity_to_probability(score, rare_base_rates[bird], rare_centers[bird], rare_sharpnesses[bird])
            final_preds[bird] = float(animal_p[4] * p)

        batch_preds.append(final_preds)

    return batch_preds

In [ ]:
species_list = ['1161364', '116570', '1176823', '1491113', '1595929', '209233', '22930', '22956']
species_list += ['22961', '22967', '22973', '22983', '22985', '23150', '23154', '23158', '23176']
species_list += ['23724', '24279', '24285', '24287', '24321', '244024', '25073', '25092', '25214']
species_list += ['326272', '41970', '43435', '47144', '47158son01', '47158son02', '47158son03']
species_list += ['47158son04', '47158son05', '47158son06', '47158son07', '47158son08']
species_list += ['47158son09', '47158son10', '47158son11', '47158son12', '47158son13']
species_list += ['47158son14', '47158son15', '47158son16', '47158son17', '47158son18']
species_list += ['47158son19', '47158son20', '47158son21', '47158son22', '47158son23']
species_list += ['47158son24', '47158son25', '476521', '516975', '517063', '555123', '555145']
species_list += ['555146', '64898', '65377', '65380', '66971', '67107', '67252', '70711', '738183']
species_list += ['74113', '74580', '760266', 'ashgre1', 'astcra1', 'bafcur1', 'baffal1', 'banana']
species_list += ['barant1', 'batbel1', 'baymac', 'bbwduc', 'bcwfin2', 'bkcdon', 'bkhpar']
species_list += ['blchaw1', 'blheag1', 'blttit1', 'bncfly', 'bobfly1', 'brcmar1', 'brnowl']
species_list += ['bucmot4', 'bucpar', 'bufpar', 'bunibi1', 'burowl', 'camfli1', 'chacha1']
species_list += ['chbmoc1', 'chobla1', 'chvcon1', 'cibspi1', 'coffal1', 'compau', 'compot1']
species_list += ['crbthr1', 'crebec1', 'dwatin1', 'epaori4', 'eulfly1', 'fabwre1', 'fepowl']
species_list += ['ficman1', 'flawar1', 'fotfly', 'fusfly1', 'gilhum1', 'giwrai1', 'glteme1']
species_list += ['grasal3', 'greani1', 'greant1', 'greela', 'grekis', 'grepot1', 'gretho2']
species_list += ['greyel', 'grfdov1', 'grhtan1', 'gycwor1', 'horscr1', 'houspa', 'hyamac1']
species_list += ['larela1', 'lesela1', 'lesgrf1', 'limpki', 'linwoo1', 'litcuc2', 'litnig1']
species_list += ['mabpar', 'magant1', 'magtan2', 'masgna1', 'nacnig1', 'ocecra1', 'oliwoo1']
species_list += ['orbtro3', 'orwpar', 'osprey', 'pabspi1', 'palhor3', 'paltan1', 'phecuc1']
species_list += ['picpig2', 'pirfly1', 'plasla1', 'platyr1', 'plcjay1', 'pluibi1', 'purjay1']
species_list += ['pvttyr1', 'ragmac1', 'rebscy1', 'recfin1', 'redjun', 'relser1', 'rinkin1']
species_list += ['rivwar1', 'roahaw', 'rubthr1', 'rufcac2', 'rufcas2', 'rufgna3', 'rufhor2']
species_list += ['rufnig1', 'ruftho1', 'ruftof1', 'rumfly1', 'ruther1', 'rutjac1', 'sabspa1']
species_list += ['saffin', 'saytan1', 'scadov1', 'schpar1', 'scther1', 'shcfly1', 'shshaw']
species_list += ['shtnig1', 'sibtan2', 'smbani', 'smbtin1', 'sobcac1', 'sobtyr1', 'socfly1']
species_list += ['sofspi1', 'souant1', 'soulap1', 'souscr1', 'spbant3', 'spispi1', 'sptnig1']
species_list += ['squcuc1', 'stbwoo2', 'strcuc1', 'strher2', 'strowl1', 'swthum1', 'swtman1']
species_list += ['tattin1', 'thlwre1', 'toctou1', 'trokin', 'trsowl', 'undtin1', 'varant1']
species_list += ['watjac1', 'wesfie1', 'wfwduc1', 'whbant2', 'whbwar2', 'whiwoo1', 'whlspi1']
species_list += ['whnjay1', 'whtdov', 'whwpic1', 'y00678', 'yebcar', 'yebela1', 'yecmac', 'yecpar']
species_list += ['yehcar1', 'yeofly1']

In [ ]:
def preprocess_chunk(chunk, sr=32000):
    mel_spec = librosa.feature.melspectrogram(
        y=chunk,
        sr=sr,
        n_mels=128
    )
    mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min())
    
    return np.expand_dims(mel_spec, axis=-1)

In [ ]:
rows = []
SR = 32000
CHUNK_SECONDS = 5
chunk_len = SR * CHUNK_SECONDS

for filename in os.listdir(TEST_DIR):
    if not filename.endswith(".ogg"):
        continue

    file_path = os.path.join(TEST_DIR, filename)
    file_id = filename.replace(".ogg", "")

    audio, sr = librosa.load(file_path, sr=SR)

    mel_specs = []
    row_ids = []

    for start_sample in range(0, len(audio), chunk_len):
        chunk = audio[start_sample:start_sample + chunk_len]

        if len(chunk) < chunk_len:
            continue

        end_seconds = (start_sample // SR) + CHUNK_SECONDS
        row_id = f"{file_id}_{end_seconds}"

        mel_specs.append(preprocess_chunk(chunk, sr=SR))
        row_ids.append(row_id)

    if len(mel_specs) == 0:
        continue

    X_file = np.array(mel_specs)

    preds_list = predict_batch(X_file)

    for row_id, preds in zip(row_ids, preds_list):
        row = {"row_id": row_id}

        for species in species_list:
            row[species] = float(preds.get(str(species), 0.0))

        rows.append(row)

submission = pd.DataFrame(rows, columns=["row_id"] + species_list)
submission.to_csv("submission.csv", index=False)